In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

This synthetic dataset is modeled after an existing milling machine and consists of 10 000 data points from a stored as rows with 14 features in columns


1. UID: unique identifier ranging from 1 to 10000
2. product ID: consisting of a letter L, M, or H for low (50% of all products), medium (30%) and high (20%) as product quality variants and a variant-specific serial number
3. type: just the product type L, M or H from column 2
4. air temperature [K]: generated using a random walk process later normalized to a standard deviation of 2 K around 300 K
5. process temperature [K]: generated using a random walk process normalized to a standard deviation of 1 K, added to the air temperature plus 10 K.
6. rotational speed [rpm]: calculated from a power of 2860 W, overlaid with a normally distributed noise
7. torque [Nm]: torque values are normally distributed around 40 Nm with a SD = 10 Nm and no negative values.
8. tool wear [min]: The quality variants H/M/L add 5/3/2 minutes of tool wear to the used tool in the process.
9. a 'machine failure' label that indicates, whether the machine has failed in this particular datapoint for any of the following failure modes are true.


The machine failure consists of five independent failure modes


1. tool wear failure (TWF): the tool will be replaced of fail at a randomly selected tool wear time between 200 - 240 mins (120 times in our dataset). At this point in time, the tool is replaced 69 times, and fails 51 times (randomly assigned).
2. heat dissipation failure (HDF): heat dissipation causes a process failure, if the difference between air- and process temperature is below 8.6 K and the tools rotational speed is below 1380 rpm. This is the case for 115 data points.
3. power failure (PWF): the product of torque and rotational speed (in rad/s) equals the power required for the process. If this power is below 3500 W or above 9000 W, the process fails, which is the case 95 times in our dataset.
4. overstrain failure (OSF): if the product of tool wear and torque exceeds 11,000 minNm for the L product variant (12,000 M, 13,000 H), the process fails due to overstrain. This is true for 98 datapoints.
5. random failures (RNF): each process has a chance of 0,1 % to fail regardless of its process parameters. This is the case for only 5 datapoints, less than could be expected for 10,000 datapoints in our dataset.

If at least one of the above failure modes is true, the process fails and the 'machine failure' label is set to 1. 

In [ ]:
df=pd.read_csv("/kaggle/input/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv")

In [ ]:
# Displaying dataset to understand the structure
df

In [ ]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
Which data points shows Machine Failure??</h3>

In [ ]:
df[df["Machine failure"]==1]

<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
DATA PREPROCESSING</h3>

In [ ]:
df.info()

In [ ]:
# Checking for null values and duplicates

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# Understanding data types and unique values per column

In [ ]:
df.dtypes

In [ ]:
df.nunique()

In [ ]:
df['Type'].value_counts()

In [ ]:
df.drop(columns=['UDI','Product ID'],inplace=True)

<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
Feature Engineering
</h3>

In [ ]:
# Creating new feature: temperature difference between process and air
df['temperature_difference']=df['Process temperature [K]']-df['Air temperature [K]']

In [ ]:
# Creating new feature: mechanical power using torque and rotational speed
df['Mechanical Power [W]']=np.round((df['Torque [Nm]']*df['Rotational speed [rpm]']* 2 * np.pi) / 60,4)

In [ ]:
df

<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
Statistical Description
</h3>

In [ ]:
df.describe().T


<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
Exploratory Data Analysis 
</h3>

# 1) Plotting the Distribution of Machine Types

#### Helps you to see how many machines belong to each type (L, M, H)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
sns.countplot(x='Type', data=df)
plt.title('Distribution of Machine Types')
plt.xlabel('Machine Type')
plt.ylabel('Count')
plt.show()


# 2) Visualizing Failure Distribution Across Product Types
#### Shows how failures are spread across types — are some types failing more?

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Type', hue='Machine failure', data=df)
plt.title('Machine Failure Distribution Across Types')
plt.xlabel('Machine Type')
plt.ylabel('Count')
plt.legend(title='Failure')
plt.show()



# 3) Plotting Feature Distributions to Observe Patterns or Anomalies

In [ ]:
cols = ['Torque [Nm]', 'Rotational speed [rpm]', 'temperature_difference', 'Mechanical Power [W]']

for col in cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # 1 row, 2 columns

    # Histogram with KDE
    sns.histplot(data=df, x=col, kde=True, ax=axes[0])
    axes[0].set_title(f"{col} Distribution")

    # Boxplot
    sns.boxplot(data=df, x=col, ax=axes[1])
    axes[1].set_title(f"{col} - Outlier Check")

    plt.tight_layout()
    plt.show()

# 4) Pairplot for Feature Relationships
#### Shows interaction between features colored by failure

In [ ]:
sns.pairplot(df[['Torque [Nm]', 'Rotational speed [rpm]', 'temperature_difference','Mechanical Power [W]', 'Machine failure']], hue='Machine failure')
plt.show()



<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
 Correlation of features
</h3>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Checking correlation between numerical features using a heatmap
corr_matrix=df.corr(numeric_only=True)
plt.figure(figsize=(8,5))
sns.heatmap(corr_matrix,annot=True,cmap='cividis',fmt=".2f", linewidths=0.5)

In [ ]:
# Checking correlation between different failure using a heatmap

In [ ]:
target=df.iloc[:,[6,7,8,9,10,11]]
target_mat=target.corr()
sns.heatmap(target_mat,annot=True,cmap="cividis",fmt=".4f",linewidth=0.5)

Tool wear failure (TWF), heat dissipation failure (HDF),power failure (PWF),overstrain failure (OSF) and random failures (RNF) shows more positive correlation with target variable i.e. machine failure. Thus dropping columns 'TWF','HDF','PWF','OSF','RNF'.


In [ ]:
df.drop(columns=['TWF','HDF','PWF','OSF','RNF'],inplace=True)

In [ ]:
df.sample(3)



<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
  Encoding Columns
</h3>

In [ ]:
# Label encoding categorical variables (column- Type)
from sklearn.preprocessing import LabelEncoder
df['Type'] = LabelEncoder().fit_transform(df['Type'])


<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
FEATURE SCALING
</h3>

In [ ]:
# Scaling numerical features using StandardScaler for model compatibility
from sklearn.preprocessing import StandardScaler
scale=StandardScaler()
data=pd.DataFrame(scale.fit_transform(df),columns=df.columns,index=df.index)

In [ ]:
data.sample(10)


<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
 Splitting data into features (X) and target (y)

</h3>

In [ ]:
Y=df.pop("Machine failure")
X=df

In [ ]:
X

In [ ]:
Y


<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
Handling Imbalanced Data
</h3>

In [ ]:
# SMOTE- Synthetic Minority Over-sampling Technique 
# SMOTE is a method for handling imbalanced datasets in machine learning. 
# Goal: To increase the number of instances in the minority class by creating synthetic samples. 
# How it works: SMOTE generates new examples by interpolating between existing minority class instances and their nearest neighbors.
# This helps the model learn better about the minority class and improves its performance on imbalanced datasets. 


In [ ]:
# print distribution of class before SMOTE
from collections import Counter

counts = Counter(Y)
print(counts)


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, Y)


In [ ]:
# print distribution of class AFTER SMOTE

from collections import Counter

counts = Counter(y_resampled )
print(counts)



<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
Train Test Split
</h3>

In [ ]:
#Performing train-test split

from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X_resampled,y_resampled,test_size=0.1)


<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
 Let's see which algorithm gives best accuracy
</h3>

In [ ]:
# Importing machine learning models

from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression,LogisticRegressionCV,SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier,AdaBoostClassifier,BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(),
        'Logistic Regression CV': LogisticRegressionCV(),
    'SGD': SGDClassifier(),
    
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'AdaBoost': AdaBoostClassifier(),
    'Bagging': BaggingClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'Support Vector Machine': SVC(),
    'K-Nearest Neighbors': KNeighborsClassifier()
}

In [ ]:
# Creating a function to fit models on our dataset and check which model gives highest accuracy

def evaluate_model(X_train,X_test,Y_train,Y_test):
    result=[]
    for name, model in models.items():
        model.fit(X_train,Y_train)
        y_pred=model.predict(X_test)
        acc=accuracy_score(Y_test,y_pred)
        result.append((name,acc))
    # Sort models by accuracy
    result.sort(key=lambda x: x[1], reverse=True)
    return result
    
        

In [ ]:
results = evaluate_model(X_train,X_test,Y_train,Y_test)
    
print("Model Performance:")
for name, acc in results:
    print(f"{name}: {acc:.6f}")

 Here, we can clearly see Random Forest is yielding best accuracy from all algorithms.
 So,we will further tune the parameters to get the better performance 

In [ ]:
 RF=RandomForestClassifier(class_weight='balanced')

In [ ]:
RF.fit(X_train,Y_train)
y_pred=RF.predict(X_test)
acc=accuracy_score(Y_test,y_pred)

In [ ]:
acc

In [ ]:
# Lets check model performances using accuracy, precision, recall, and F1-score

In [ ]:
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    roc_auc_score, 
    RocCurveDisplay, 
    precision_recall_curve, 
    PrecisionRecallDisplay
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Classification Report
    print(f"--------- {model_name} Classification Report ------ \n\n")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')
    plt.title(f"{model_name} - Confusion Matrix")
    plt.show()

    # ROC Curve
    roc_auc = roc_auc_score(y_test, y_prob)
    RocCurveDisplay.from_predictions(y_test, y_prob)
    plt.title(f"{model_name} - ROC Curve (AUC = {roc_auc:.2f})")
    plt.show()

    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    PrecisionRecallDisplay(precision=precision, recall=recall).plot()
    plt.title(f"{model_name} - Precision-Recall Curve")
    plt.show()


In [ ]:
evaluate_model(RF, X_test, Y_test, model_name="Random Forest")


In [ ]:
importances = RF.feature_importances_
feature_names = X.columns

# Create a DataFrame
feature_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot
plt.figure(figsize=(8, 5))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df, palette='viridis')
plt.title('Feature Importance - Random Forest')
plt.tight_layout()
plt.show()


<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
 Found it useful? Upvote 🙌
</h3>